In [2]:
from nba_api.stats.static import teams
from nba_api.stats.endpoints import commonteamroster
from sqlalchemy import create_engine, inspect
import pandas as pd
import random
import time
from dotenv import load_dotenv
import os

# helper functions
def normalize_for_postgres(df):
    """Normalize DataFrame column names for PostgreSQL"""
    df = df.copy()
    df.columns = df.columns.str.lower().str.replace(' ', '_')
    return df
    
load_dotenv()
DATABASE_URL = os.getenv("DATABASE_URL")
engine = create_engine(DATABASE_URL)

# --- NEW: INITIALIZE SEEN_PLAYER_IDS FROM DATABASE ---
print("Checking database for existing players...")
inspector = inspect(engine)
seen_player_ids = set()

if 'players' in inspector.get_table_names():
    # Fetch IDs already in the database to avoid duplicates/errors
    existing_ids_df = pd.read_sql("SELECT player_id FROM players", engine)
    seen_player_ids = set(existing_ids_df['player_id'].tolist())
    print(f"Found {len(seen_player_ids)} existing players in the database.")
else:
    print("Table 'players' does not exist yet. Starting fresh.")

# Set target seasons
seasons = ['2024-25']

nba_teams = teams.get_teams()
print(f"Found {len(nba_teams)} teams in total\n")

all_players = []
team_count = 0

for team in nba_teams:
    team_id = team['id']
    team_name = team['full_name']
    team_count += 1
    
    print(f"[{team_count}/{len(nba_teams)}] Processing team: {team_name}")
    
    try:   
        for season in seasons:
            time.sleep(random.uniform(1.5, 3.0)) # Avoid rate limiting
            
            try:
                roster = commonteamroster.CommonTeamRoster(team_id=team_id, season=season)
                roster_df = roster.get_data_frames()[0]
                
                # Filter out players we've already seen (in DB or earlier in this loop)
                new_players_df = roster_df[~roster_df['PLAYER_ID'].isin(seen_player_ids)]
                
                if len(new_players_df) > 0:
                    new_players_df = new_players_df[['PLAYER_ID', 'PLAYER']].copy()
                    new_players_df.rename(columns={'PLAYER': 'PLAYER_NAME'}, inplace=True)
                    
                    all_players.extend(new_players_df.to_dict('records'))
                    seen_player_ids.update(new_players_df['PLAYER_ID'].tolist())
                    
                    print(f"  ✓ Found {len(new_players_df)} new unique players")
                else:
                    print(f"  - No new players")
                    
            except Exception as season_error:
                print(f"  ⚠ Error fetching roster: {str(season_error)}")
                continue
        
    except Exception as e:
        print(f"  ⚠ Error processing {team_name}: {str(e)}")
    
    print("-" * 40)

# --- FINAL UPDATE ---
if all_players:
    players_df = pd.DataFrame(all_players)
    players_df = players_df.drop_duplicates(subset=['PLAYER_ID'])
    players_df = normalize_for_postgres(players_df)

    print(f"\nPushing {len(players_df)} new players to database...")
    # CHANGED: 'append' instead of 'replace'
    players_df.to_sql('players', engine, if_exists='append', index=False)
    print("✅ Successfully appended new players.")
else:
    print("\n🙌 No new players found to add. Your 'players' table is already up to date!")

Checking database for existing players...
Found 1645 existing players in the database.
Found 30 teams in total

[1/30] Processing team: Atlanta Hawks
  ✓ Found 3 new unique players
----------------------------------------
[2/30] Processing team: Boston Celtics
  ✓ Found 2 new unique players
----------------------------------------
[3/30] Processing team: Cleveland Cavaliers
  ✓ Found 3 new unique players
----------------------------------------
[4/30] Processing team: New Orleans Pelicans
  ✓ Found 4 new unique players
----------------------------------------
[5/30] Processing team: Chicago Bulls
  ✓ Found 3 new unique players
----------------------------------------
[6/30] Processing team: Dallas Mavericks
  - No new players
----------------------------------------
[7/30] Processing team: Denver Nuggets
  ✓ Found 4 new unique players
----------------------------------------
[8/30] Processing team: Golden State Warriors
  ✓ Found 3 new unique players
-----------------------------------